# Assignment 4: Inside the Residual Stream
## Stats 292 — Statistical Models of Text and Language
### Prof. David Donoho · Stanford University · Spring 2026

**Published:** May 11, 2026 · **Due:** May 21, 2026, 11:59 PM

---

Submit this notebook with all cells executed. Figures should be labeled.
Written derivations go in Markdown cells (LaTeX supported).
**Approximate time:** 8–12 hours for Parts 1–4; Part 5 is optional.

## Setup

Run these cells once at the start of each Colab session.

In [ ]:
# Install dependencies (~60 s on Colab free tier)
!pip install transformer_lens datasets -q

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from transformer_lens.model_bridge import TransformerBridge

In [ ]:
# Downloads ~500 MB of GPT-2 weights from HuggingFace (~1 min on Colab)
bridge = TransformerBridge.boot_transformers("gpt2", device="cpu")
bridge.enable_compatibility_mode()   # fold LN, center weights; enables legacy hook names

cfg = bridge.cfg
print(f"Model:   {cfg.model_name}")
print(f"d_model: {cfg.d_model}  |  n_layers: {cfg.n_layers}  |  n_heads: {cfg.n_heads}")
print(f"d_head:  {cfg.d_head}   |  d_vocab:  {cfg.d_vocab}")

In [ ]:
# Clone Marks & Tegmark dataset repository
!git clone https://github.com/saprmarks/geometry-of-truth 2>/dev/null || echo "already cloned"

cities = pd.read_csv("geometry-of-truth/datasets/cities.csv")
sp_en  = pd.read_csv("geometry-of-truth/datasets/sp_en_trans.csv")

print("cities columns:", cities.columns.tolist())
print(cities.head(3))
print(f"\ncities: {len(cities)} rows | labels: {cities['label'].value_counts().to_dict()}")
print(f"sp_en:  {len(sp_en)} rows  | labels: {sp_en['label'].value_counts().to_dict()}")

---
## Part 1 — Architecture as Linear Algebra (warm-up, ~1 hour)

All answers should reference specific tensor shapes or parameter counts from GPT-2 small.

**1.1** Report: $d_{\text{model}}$, number of layers $L$, number of attention heads $H$,
head dimension $d_h = d_{\text{model}} / H$, vocabulary size $V$.

In [ ]:
# 1.1
cfg = bridge.cfg
print(f"d_model  = {cfg.d_model}")
print(f"n_layers = {cfg.n_layers}")
print(f"n_heads  = {cfg.n_heads}")
print(f"d_head   = {cfg.d_head}")
print(f"d_vocab  = {cfg.d_vocab}")

**1.2** The residual stream at position $i$ after layer $\ell$ is a vector
$x_i^{(\ell)} \in \mathbb{R}^{d_{\text{model}}}$. Every attention head and MLP
*adds* its output to this stream. Write down the update equation for one layer
in terms of the attention sublayer output $a^{(\ell)}$ and MLP sublayer output $m^{(\ell)}$.

**Answer 1.2:**

<!-- Write your equation here using LaTeX -->

**1.3** For a single attention head: what are the shapes of $W_Q$, $W_K$, $W_V$,
$W_O$? What is the rank of the matrix $W_V W_O$ (the OV circuit)? What does that
rank imply about how much information one head can write per token?

In [ ]:
# 1.3 — Inspect attention weight shapes for layer 0
for name, param in bridge.named_parameters():
    if "blocks.0" in name and any(x in name for x in ["W_Q", "W_K", "W_V", "W_O"]):
        print(f"{name}: {param.shape}")

# Then: what is rank(W_V @ W_O)?

**Answer 1.3 (rank and interpretation):**

<!-- Write here -->

**1.4** The unembedding matrix $W_U \in \mathbb{R}^{V \times d_{\text{model}}}$
maps the final residual stream to logits over the vocabulary. Is $W_U$ full rank?
What is its rank?

In [ ]:
# 1.4 — Unembedding matrix rank
W_U = bridge.W_U    # shape (d_vocab, d_model)
print(f"W_U shape: {W_U.shape}")
print(f"max possible rank: min(V, d_model) = {min(W_U.shape[0], W_U.shape[1])}")
# Note: exact matrix_rank on a 50k x 768 matrix is slow; reason from dimensions instead

**Answer 1.4 (rank and reasoning):**

<!-- Write here -->

---
## Part 2 — The Logit Lens (~3 hours)

**Background.** The logit lens applies $W_U$ to the residual stream at every layer:

$$\hat{p}^{(\ell)} = \text{softmax}\!\left(W_U \cdot \text{LayerNorm}\!\left(x^{(\ell)}\right)\right)$$

**API reminder:** `bridge.unembed(bridge.ln_final(cache["resid_post", ell]))` returns the
$(\text{batch} \times \text{seq\_len} \times \text{vocab})$ logit tensor at layer $\ell$.
Use `bridge.to_str_tokens(text)` to decode token strings.

**2.1** Choose a sentence of 10–15 tokens on a topic of your choice. For each
token position and each layer $\ell = 0, 1, \ldots, L{-}1$, compute the top-1 predicted
token under the logit lens. Produce a heatmap: rows = layers, columns = token
positions, cell content = top-1 token string.

In [ ]:
# 2.1 — Logit lens heatmap
MY_SENTENCE = "The Eiffel Tower is located in Paris, France."   # ← replace with your sentence

_, cache = bridge.run_with_cache(MY_SENTENCE)
tokens = bridge.to_str_tokens(MY_SENTENCE)
n_layers = bridge.cfg.n_layers

# Collect top-1 token at each (layer, position)
top1 = []
for ell in range(n_layers):
    resid = cache["resid_post", ell][0]                           # (seq_len, d_model)
    logits = bridge.unembed(bridge.ln_final(resid))               # (seq_len, d_vocab)
    top1.append([bridge.tokenizer.decode([t]) for t in logits.argmax(dim=-1).tolist()])

# TODO: plot as heatmap — rows = layers, columns = positions, text = top1[ell][pos]
# Hint: use plt.imshow + ax.text(), or build a DataFrame and display it
print(f"Input tokens:  {tokens}")
print(f"Layer 0  top-1: {top1[0]}")
print(f"Layer 11 top-1: {top1[-1]}")

**2.2** Pick 3–4 content tokens in your sentence (nouns, verbs — not "the", "of").
For each, plot: (a) the logit of the correct next token vs. layer, and
(b) the rank of the correct next token vs. layer. At what layer does each prediction
"lock in"? Is this consistent across token types?

In [ ]:
# 2.2 — Prediction lock-in curves
# Map: token position → correct next token string
# Example: if tokens[3] is "Tower" and tokens[4] is " is", then correct_next[3] = " is"
content_positions = {
    # pos: "correct next token"   ← fill in for your sentence
}

fig, axes = plt.subplots(len(content_positions), 2, figsize=(12, 3 * len(content_positions)))
if len(content_positions) == 1:
    axes = axes[None]   # ensure 2D

for row, (pos, next_tok) in enumerate(content_positions.items()):
    tok_id = bridge.tokenizer.encode(next_tok)[0]
    logit_curve, rank_curve = [], []
    for ell in range(n_layers):
        resid_pos = cache["resid_post", ell][0, pos]     # (d_model,)
        lgts = bridge.unembed(bridge.ln_final(resid_pos.unsqueeze(0).unsqueeze(0)))[0, 0]
        logit_curve.append(lgts[tok_id].item())
        rank_curve.append((lgts > lgts[tok_id]).sum().item())
    axes[row, 0].plot(logit_curve); axes[row, 0].set_title(f"pos {pos}: logit of '{next_tok}'")
    axes[row, 1].plot(rank_curve);  axes[row, 1].set_title(f"pos {pos}: rank of '{next_tok}'")
    axes[row, 0].set_xlabel("layer"); axes[row, 1].set_xlabel("layer")

plt.tight_layout(); plt.show()

**2.3** Compute for each layer $\ell$:

$$D_\text{KL}\!\left(\hat{p}^{(L-1)} \;\big\|\; \hat{p}^{(\ell)}\right)$$

Plot this vs. layer. What happens between the embedding layer (layer $-1$,
i.e. the raw token embedding) and layer 0?

In [ ]:
# 2.3 — KL divergence from final-layer distribution
import torch.nn.functional as F

# Get final-layer distribution (reference)
# cache["embed"] gives the raw token embeddings (layer -1 equivalent)
final_logits = bridge.unembed(bridge.ln_final(cache["resid_post", n_layers - 1][0]))
p_final = final_logits.softmax(dim=-1)   # (seq_len, vocab)

kl_by_layer = []
for ell in range(n_layers):
    resid = cache["resid_post", ell][0]
    logits_ell = bridge.unembed(bridge.ln_final(resid))
    p_ell = logits_ell.softmax(dim=-1)
    # KL(p_final || p_ell) averaged over positions
    kl = F.kl_div(p_ell.log(), p_final, reduction="batchmean").item()
    kl_by_layer.append(kl)

# Also compute for the raw embedding (layer -1)
embed = cache["hook_embed"][0]   # raw token embedding, shape (seq_len, d_model)
logits_embed = bridge.unembed(bridge.ln_final(embed))
p_embed = logits_embed.softmax(dim=-1)
kl_embed = F.kl_div(p_embed.log(), p_final, reduction="batchmean").item()

layers = list(range(n_layers))
plt.figure(figsize=(8, 4))
plt.plot([-1] + layers, [kl_embed] + kl_by_layer, marker="o")
plt.xlabel("layer"); plt.ylabel("KL divergence from final layer")
plt.title("Logit lens: KL divergence vs. layer")
plt.axvline(0, color="gray", linestyle="--", alpha=0.5, label="layer 0")
plt.legend(); plt.show()

**2.4 (Mathematical question)** The logit lens applies the *final* unembedding
matrix $W_U$ to intermediate representations. What implicit assumption does this
make? Under what conditions is this assumption justified? What would the tuned lens
(Belrose et al.) do differently, and why?

**Answer 2.4:**

<!-- Write here -->

**2.5** Try your analysis on a second sentence that contains an unusual or rare token
(a proper name, a technical term). Does the rare token behave differently in the
logit lens than common tokens? Relate your observation to nostalgebraist's "plasma" example.

In [ ]:
# 2.5 — Rare token analysis
RARE_SENTENCE = "..."   # ← choose a sentence with at least one rare/proper-name token

_, cache_rare = bridge.run_with_cache(RARE_SENTENCE)
tokens_rare = bridge.to_str_tokens(RARE_SENTENCE)
print("Tokens:", tokens_rare)

# TODO: reproduce your 2.1 / 2.2 analysis on this sentence
# Compare lock-in behavior for the rare token vs. common tokens

---
## Part 3 — Mass-Mean Probe for Factual Truth (~3 hours)

**Background.** The mass-mean probe estimates the truth direction in residual stream space:

$$\hat{\theta}^{(\ell)} = \frac{1}{N^+}\sum_{i:\,y_i=1} x_i^{(\ell)} \;-\;
\frac{1}{N^-}\sum_{j:\,y_j=0} x_j^{(\ell)}$$

where $x_i^{(\ell)}$ is the residual stream at the **final token position** of statement $i$, layer $\ell$.

**Note:** The `cities` and `sp_en_trans` CSV files have a `statement` column with pre-formatted
statements ready to use.

**3.1** Load the `cities` dataset. Use the first 200 rows as training set and the
next 100 as test set. Verify that the label distribution is balanced.

In [ ]:
# 3.1 — Data preparation
train_df = cities.iloc[:200].reset_index(drop=True)
test_df  = cities.iloc[200:300].reset_index(drop=True)

print(f"Train: {len(train_df)} rows | label counts: {train_df['label'].value_counts().to_dict()}")
print(f"Test:  {len(test_df)} rows  | label counts: {test_df['label'].value_counts().to_dict()}")
print("\nExample statements:")
print(train_df[["statement", "label"]].head(4).to_string())

**3.2** For each statement in your training set, run the model and cache
`resid_post` at every layer. Extract the activation at the final token position.
Result: matrix $X^{(\ell)} \in \mathbb{R}^{200 \times d_{\text{model}}}$ for each $\ell$.

In [ ]:
# 3.2 — Extract activations  (takes ~5 min on Colab CPU for 200 statements)
# To save time during development, test with n=20 first, then scale to 200.

def extract_final_token_acts(df, desc=""):
    """Returns (acts_by_layer, labels) where acts_by_layer[ell] is (N, d_model)."""
    acts = {ell: [] for ell in range(bridge.cfg.n_layers)}
    labels = []
    for i, row in df.iterrows():
        _, cache_i = bridge.run_with_cache(row["statement"])
        for ell in range(bridge.cfg.n_layers):
            acts[ell].append(cache_i["resid_post", ell][0, -1, :].detach())
        labels.append(row["label"])
        if (i + 1) % 50 == 0:
            print(f"  {desc}: processed {i+1}/{len(df)}")
    return {ell: torch.stack(acts[ell]) for ell in acts}, torch.tensor(labels, dtype=torch.float)

print("Extracting train activations...")
X_train, y_train = extract_final_token_acts(train_df, "train")
print("Extracting test activations...")
X_test,  y_test  = extract_final_token_acts(test_df,  "test")
print(f"Done. X_train[0].shape = {X_train[0].shape}")

**3.3** For each layer $\ell$, compute $\hat{\theta}^{(\ell)}$ and normalize to unit vector $\hat{r}^{(\ell)}$.

In [ ]:
# 3.3 — Mass-mean directions
r_hat = {}   # r_hat[ell] = unit-norm probe direction at layer ell
for ell in range(bridge.cfg.n_layers):
    X = X_train[ell]                                  # (200, d_model)
    y = y_train                                       # (200,)
    mu_pos = X[y == 1].mean(dim=0)
    mu_neg = X[y == 0].mean(dim=0)
    theta = mu_pos - mu_neg
    r_hat[ell] = theta / theta.norm()

**3.4** For each layer, classify the test set using signed projection. Plot accuracy vs. layer.

In [ ]:
# 3.4 — Probe accuracy vs. layer
accuracies = []
for ell in range(bridge.cfg.n_layers):
    X = X_test[ell]
    projections = X @ r_hat[ell]                      # (100,) signed scalar per statement
    # Choose threshold c on the training set
    train_proj = X_train[ell] @ r_hat[ell]
    c = (train_proj[y_train == 1].mean() + train_proj[y_train == 0].mean()) / 2
    preds = (projections > c).float()
    acc = (preds == y_test).float().mean().item()
    accuracies.append(acc)

best_layer = int(np.argmax(accuracies))
print(f"Best layer: {best_layer}, accuracy: {accuracies[best_layer]:.3f}")

plt.figure(figsize=(8, 4))
plt.plot(accuracies, marker="o")
plt.axhline(0.5, color="gray", linestyle="--", label="chance")
plt.xlabel("layer"); plt.ylabel("test accuracy")
plt.title("Mass-mean probe accuracy vs. layer (cities dataset)")
plt.legend(); plt.show()

**3.5** At the best layer from 3.4, also fit an $\ell_2$-regularized logistic regression probe. Does LR outperform mass-mean?

In [ ]:
# 3.5 — Logistic regression baseline
from sklearn.linear_model import LogisticRegression

X_tr = X_train[best_layer].numpy()
X_te = X_test[best_layer].numpy()
y_tr = y_train.numpy().astype(int)
y_te = y_test.numpy().astype(int)

lr = LogisticRegression(C=1.0, max_iter=1000)
lr.fit(X_tr, y_tr)
lr_acc = lr.score(X_te, y_te)

print(f"Layer {best_layer}:")
print(f"  Mass-mean accuracy: {accuracies[best_layer]:.3f}")
print(f"  Logistic regression: {lr_acc:.3f}")

**3.6 (Mathematical question)** Show that the mass-mean probe is equivalent to
Fisher's Linear Discriminant (LDA) direction when the two class covariance matrices
are equal and proportional to the identity. What does superposition (non-identity
covariance, many features packed into $\mathbb{R}^{d_{\text{model}}}$) do to this
equivalence? In which direction do you expect LR and mass-mean to diverge, and
why might the mass-mean direction be *more* causally relevant despite its
statistical sub-optimality?

**Answer 3.6:**

<!-- Write your derivation and discussion here -->

**3.7** Evaluate your `cities`-trained mass-mean probe on `sp_en_trans` without
retraining. Does the probe transfer? What does transfer (or its absence) imply?

In [ ]:
# 3.7 — Transfer to sp_en_trans
print("Extracting sp_en activations...")
X_sp, y_sp = extract_final_token_acts(sp_en.iloc[:100], "sp_en")

# Evaluate cities probe (best_layer) on sp_en test data
proj_sp = X_sp[best_layer] @ r_hat[best_layer]
# Use the same threshold c from Part 3.4
preds_sp = (proj_sp > c).float()
transfer_acc = (preds_sp == y_sp[:100]).float().mean().item()
print(f"Transfer accuracy (cities probe → sp_en): {transfer_acc:.3f}")

**Answer 3.7 (interpretation):**

<!-- Write here -->

---
## Part 4 — Causal Intervention: Is the Direction Causal? (~2 hours)

**Background.** The *directional ablation* hook removes $\hat{r}^{(\ell^*)}$ from the
residual stream at layer $\ell^*$:

$$x \;\leftarrow\; \left(I - \hat{r}^{(\ell^*)}\hat{r}^{(\ell^*)T}\right) x$$

Use `bridge.run_with_hooks(text, fwd_hooks=[(hook_name, hook_fn)])` where
`hook_name = f"blocks.{best_layer}.hook_resid_post"`.

**4.1** Implement directional ablation as a TransformerLens hook.
The hook function receives the activation tensor and returns the modified tensor.

In [ ]:
# 4.1 — Directional ablation hook
def make_ablation_hook(r):
    """Returns a hook that projects out the component along unit vector r."""
    def hook_fn(value, hook):
        # value: (batch, seq_len, d_model)
        proj = (value @ r).unsqueeze(-1) * r    # component along r
        return value - proj
    return hook_fn

def make_addition_hook(r, alpha=20.0):
    """Returns a hook that adds alpha * r to the residual stream."""
    def hook_fn(value, hook):
        return value + alpha * r
    return hook_fn

hook_name = f"blocks.{best_layer}.hook_resid_post"
r = r_hat[best_layer]   # unit vector from Part 3

# Quick sanity check: run one statement with and without ablation
test_stmt = test_df.iloc[0]["statement"]
out_normal  = bridge.run_with_hooks(test_stmt)
out_ablated = bridge.run_with_hooks(test_stmt, fwd_hooks=[(hook_name, make_ablation_hook(r))])
print("Sanity check: normal output shape:", out_normal.shape)
print("              ablated output shape:", out_ablated.shape)

**4.2** For 20 *true* statements, measure log-probability of " True" vs. " False"
with and without ablation. Does ablation shift the model toward "False"?

In [ ]:
# 4.2 — Ablation on true statements
true_stmts = test_df[y_test == 1]["statement"].values[:20]

true_id  = bridge.tokenizer.encode(" True")[0]
false_id = bridge.tokenizer.encode(" False")[0]

def logprob_true_minus_false(output_logits):
    """Log P(True) - Log P(False) at the last token position."""
    last = output_logits[0, -1]   # (vocab,)
    lp = torch.log_softmax(last, dim=-1)
    return (lp[true_id] - lp[false_id]).item()

scores_normal, scores_ablated = [], []
reversals = 0
for stmt in true_stmts:
    out_n = bridge.run_with_hooks(stmt)
    out_a = bridge.run_with_hooks(stmt, fwd_hooks=[(hook_name, make_ablation_hook(r))])
    sn = logprob_true_minus_false(out_n)
    sa = logprob_true_minus_false(out_a)
    scores_normal.append(sn)
    scores_ablated.append(sa)
    if sn > 0 and sa < 0:
        reversals += 1

print(f"True statements: {reversals}/20 reversed from True→False preference by ablation")
print(f"Mean log P(True)-P(False): {np.mean(scores_normal):.3f} → {np.mean(scores_ablated):.3f}")

**4.3** Repeat 4.2 for 20 *false* statements, but *add* the direction
($x \leftarrow x + \alpha \hat{r}^{(\ell^*)}$). Does addition shift toward "True"?

In [ ]:
# 4.3 — Addition on false statements
false_stmts = test_df[y_test == 0]["statement"].values[:20]

scores_normal_f, scores_added_f = [], []
reversals_f = 0
for stmt in false_stmts:
    out_n = bridge.run_with_hooks(stmt)
    out_a = bridge.run_with_hooks(stmt, fwd_hooks=[(hook_name, make_addition_hook(r, alpha=20.0))])
    sn = logprob_true_minus_false(out_n)
    sa = logprob_true_minus_false(out_a)
    scores_normal_f.append(sn)
    scores_added_f.append(sa)
    if sn < 0 and sa > 0:
        reversals_f += 1

print(f"False statements: {reversals_f}/20 reversed from False→True preference by addition")
print(f"Mean log P(True)-P(False): {np.mean(scores_normal_f):.3f} → {np.mean(scores_added_f):.3f}")

**4.4 (Mathematical question)** The operator $P_\perp = I - \hat{r}\hat{r}^T$ is
an orthogonal projection. What is its rank? What is the dimension of its null space?
Arditi et al. (2024) show the same operation can be "baked into" the weights via
$W \leftarrow W - W\hat{r}\hat{r}^T$. Show algebraically that these two procedures
produce the same output for any input $x$.

**Answer 4.4:**

<!-- Write your derivation here -->

---
## Part 5 — Geometry (Extension, ~2 hours)

*This part is optional.*

**5.1** At your best layer $\ell^*$, compute the PCA of the $300 \times d_{\text{model}}$
activation matrix (200 train + 100 test). Plot the first two principal components,
colored by true/false label.

In [ ]:
# 5.1 — PCA visualization
from sklearn.decomposition import PCA

X_all = torch.cat([X_train[best_layer], X_test[best_layer]], dim=0).numpy()
y_all = torch.cat([y_train, y_test]).numpy()

pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_all)

plt.figure(figsize=(7, 5))
for label, color, marker in [(1, "steelblue", "o"), (0, "tomato", "x")]:
    mask = y_all == label
    plt.scatter(X_2d[mask, 0], X_2d[mask, 1], c=color, marker=marker,
                alpha=0.6, label=f"label={label}")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title(f"PCA of resid_post at layer {best_layer} (cities dataset)")
plt.legend(); plt.show()
print(f"Variance explained: PC1={pca.explained_variance_ratio_[0]:.3f}, PC2={pca.explained_variance_ratio_[1]:.3f}")

**5.2** What fraction of the true/false label variance is explained by
$\hat{r}^{(\ell^*)}$ alone? Compare to the fraction explained by the first PC.

In [ ]:
# 5.2 — Variance explained by mass-mean direction vs. PC1
X_c = X_all - X_all.mean(axis=0)           # mean-center
r_np = r_hat[best_layer].numpy()

frac_mm = (X_c @ r_np).var() / np.sum(X_c**2, axis=1).mean()
frac_pc1 = pca.explained_variance_ratio_[0]

print(f"Fraction of variance explained by mass-mean direction: {frac_mm:.4f}")
print(f"Fraction explained by first PC:                        {frac_pc1:.4f}")
print(f"Cosine similarity between r_hat and PC1: {abs(float(X_c.T @ (X_c @ r_np) / np.linalg.norm(X_c @ r_np))) :.4f}")

**5.3** The mass-mean direction and the first PC may not be the same. In what sense
could the mass-mean direction be more *causally* meaningful even if it captures
less variance than the first PC?

**Answer 5.3:**

<!-- Write here -->

**5.4 (Open-ended)** The Marks & Tegmark paper found that the truth direction
becomes cleaner at larger scale (7B → 13B → 70B). Does scale matter even within
the GPT-2 family? Run your Part 3 analysis on GPT-2 medium (355M) and compare
accuracy vs. layer curves.

In [ ]:
# 5.4 — GPT-2 medium comparison
# bridge_medium = TransformerBridge.boot_transformers("gpt2-medium", device="cpu")
# bridge_medium.enable_compatibility_mode()
# Re-run extract_final_token_acts and mass-mean probe...
# Compare accuracy curves between gpt2 and gpt2-medium

---
## References

- nostalgebraist (2020). *Interpreting GPT: The Logit Lens.* LessWrong.
- Belrose et al. (2023). *Eliciting Latent Predictions from Transformers with the Tuned Lens.* arXiv:2303.08112.
- Marks & Tegmark (2023). *The Geometry of Truth.* arXiv:2310.06824.
- Arditi et al. (2024). *Refusal in Language Models Is Mediated by a Single Direction.* arXiv:2406.11717.
- Elhage et al. (2022). *Toy Models of Superposition.* Transformer Circuits Thread.
- Nanda & Bloom (2022). *TransformerLens.* github.com/TransformerLensOrg/TransformerLens.